# Expérience : les méthodes de transfert face aux références sur S1

**Objectif.** Faire tourner, sur le scénario S1 (Letter), les cinq références sans transfert et les méthodes de transfert déjà implémentées (SER, STRUT, SER*, STRUT* pour l'instant), et comparer les résultats : une méthode de transfert bat-elle les références ? À quels budgets ? Bat-elle Union-P, la référence la plus forte de la première expérience ?

**Ce que le notebook utilise.** Uniquement le code du paquet `transfer_forest` : le contrat (`Context`, `make_forest`), le chargeur de S1, les cinq références (`Src`, `Tgt`, `Union`, `Union-P`, `Oracle`) et toutes les méthodes inscrites dans `METHODS` (ici, les quatre méthodes de `methods/segev.py`).

**Ce qui est nouveau par rapport à `experience_references_s1.ipynb`.**
- Les résultats sont **enregistrés** dans `results/s1_transfert.csv` (dossier ignoré par git) : en relançant, seules les combinaisons manquantes (méthode, graine, budget) sont calculées. Quand une méthode s'ajoute à `METHODS`, il suffit de relancer : seule la nouvelle est calculée.
- Les graines sont réparties sur les cœurs de l'ordinateur.
- Les méthodes de transfert sont comparées **graine par graine** à Tgt et à Union-P (section 6).

**Ce qui est provisoire.** Deux maillons du pipeline n'existent pas encore :
- le **tirage** (test, réserve, budgets emboîtés) est repris ici de la première expérience, en suivant le protocole ; il sera remplacé par le maillon protocole ;
- la **notation** se limite à la note principale de S1, le taux d'erreur ; les autres notes (BER, F1, log loss) viendront avec `evaluate`.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split

from transfer_forest.contracts import Context, make_forest
from transfer_forest.data import load_scenarios
from transfer_forest.methods import METHODS, make_method

## 1. Le scénario S1

Letter : 20 000 lettres capitales décrites par 16 variables, 26 classes. Comme Segev et al. (2017), on coupe selon `x2bar`, la variance horizontale des pixels allumés : les lettres étroites ($x2bar \le 4$, la médiane) forment la source, les lettres étalées la cible.

In [ ]:
[scn] = load_scenarios("S1")
print(f"source : {scn.n_S} exemples   cible : {len(scn.yt)} exemples   K = {scn.K}   d = {scn.d}")

## 2. Le tirage (provisoire)

On suit le protocole :
- la cible est coupée une fois pour toutes en **test** (50 %, stratifié, graine fixe) et **réserve** ; le test ne sert qu'à noter ;
- pour chaque graine, la réserve est mélangée ; les $K$ premiers éléments couvrent les $K$ classes (sinon, à 1 %, il manque 2 à 5 lettres) ;
- le budget $p$ donne $n_T = \max(\mathrm{round}(p \cdot n_S),\, K)$ exemples étiquetés : les $n_T$ premiers de l'ordre. Les budgets sont donc **emboîtés**.

In [ ]:
# Provisoire : ces constantes et ce tirage reviendront au contrat et au maillon protocole.
TEST_SEED, SEEDS, BUDGETS = 12345, range(20), (0.01, 0.02, 0.05, 0.10, 0.20)

reserve, test = train_test_split(np.arange(len(scn.yt)), test_size=scn.test_frac,
                                 stratify=scn.yt, random_state=TEST_SEED)
Xr, yr = scn.Xt[reserve], scn.yt[reserve]
X_test, y_test = scn.Xt[test], scn.yt[test]

def reserve_order(seed):
    """Ordre de la réserve pour une graine ; les K premiers éléments couvrent les K classes."""
    perm = np.random.default_rng(seed).permutation(len(yr))
    first = np.sort(np.unique(yr[perm], return_index=True)[1])   # 1re apparition de chaque classe
    return np.concatenate([perm[first], np.delete(perm, first)])

n_T = {p: max(round(p * scn.n_S), scn.K) for p in BUDGETS}
print(f"réserve : {len(yr)}   test : {len(y_test)}   budgets n_T : {list(n_T.values())}")

## 3. Le décalage entre source et cible

Deux questions, mesurées sur la **réserve** (jamais sur le test) :
- **(a) les entrées changent-elles ?** Distribution de `x2bar`, puis, pour chaque variable, l'écart de moyennes standardisé
$$\mathrm{SMD}_j = \frac{\bar x^{T}_j - \bar x^{S}_j}{\sqrt{\big(s^{S\,2}_j + s^{T\,2}_j\big)/2}} ;$$
- **(b) les proportions de classes changent-elles ?** Pour chaque lettre, le rapport $\lambda_k = P_T(Y = k) / P_S(Y = k)$ (celui qu'utilise STRUT*) : $\lambda_k = 1$ si la lettre est aussi fréquente dans les deux domaines.

In [ ]:
j = scn.params["split_feature_idx"]                                  # colonne de x2bar
lam = (np.bincount(yr) / len(yr)) / (np.bincount(scn.ys) / scn.n_S)  # P_T(k) / P_S(k)
rank = np.argsort(lam)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
bins = np.arange(17) - 0.5
ax1.hist(scn.Xs[:, j], bins, alpha=0.6, label="source")
ax1.hist(Xr[:, j], bins, alpha=0.6, label="cible (réserve)")
ax1.set(xlabel="x2bar", ylabel="nombre d'exemples", title="(a) La variable de coupe")
ax1.legend()
ax2.bar(np.array(scn.class_names)[rank], lam[rank])
ax2.axhline(1, color="k", ls="--")
ax2.set(yscale="log", ylabel=r"$\lambda_k = P_T(k)\,/\,P_S(k)$", title="(b) Changement des proportions de classes")
plt.show()

In [ ]:
smd = (Xr.mean(0) - scn.Xs.mean(0)) / np.sqrt((scn.Xs.var(0) + Xr.var(0)) / 2)
pd.Series(smd, index=scn.feature_names).sort_values(key=abs, ascending=False).head(6).round(2)

### Ce que l'on observe

**Les entrées changent.** Par construction, les deux domaines n'ont aucune valeur de `x2bar` en commun : $x2bar \le 4$ dans la source, $\ge 5$ dans la cible ($\mathrm{SMD} = 2{,}31$). Le décalage déborde un peu sur une variable liée, `xybar` ($-0{,}66$) ; toutes les autres restent sous $0{,}35$ en valeur absolue. C'est un **décalage des covariables** concentré sur une direction de l'espace.

**Les proportions de classes changent aussi, et fortement.** $\lambda_k$ va de $0{,}08$ (X) à $41$ (O) : certaines lettres sont naturellement étalées (O, G, U, C, Q sont presque toutes dans la cible), d'autres naturellement étroites (X, A, I, T presque toutes dans la source). La source ne contient par exemple qu'une vingtaine de O. La coupe sur `x2bar` crée donc, en plus, un **décalage des classes** : $P_S(Y) \ne P_T(Y)$.

> **Regard critique.** Le protocole présente S1 comme un décalage « des entrées ». C'est incomplet : S1 combine décalage des covariables et décalage des classes. Conséquence directe pour la suite : la note de méthodes prévoyait SER* $\approx$ SER et STRUT* $\approx$ STRUT sur Letter, supposé équilibré. Ce n'est pas acquis : la correction des proportions de STRUT* a précisément de quoi agir ici.

## 4. Faire tourner les références

Pour chaque graine : on apprend la forêt source (partagée par toutes les méthodes), puis, pour chaque budget, on construit le `Context` et on fait apprendre et prédire chaque méthode. Src et Oracle ne dépendent pas du budget : une seule exécution par graine. Au total $20 \times (2 + 7 \times 5) = 740$ expériences : quelques minutes pour les références, davantage pour SER et SER*, les plus lentes (jusqu'à 15 s par entraînement au budget de 20 %). Chaque résultat est enregistré dans `results/s1_transfert.csv` : une relance ne recalcule que ce qui manque, et les graines sont réparties sur les cœurs.

### Les méthodes lancées

**Les cinq références sans transfert** (déjà décrites dans la première expérience) : `Src` (forêt source telle quelle), `Tgt` (forêt apprise sur les seuls exemples cibles étiquetés), `Union` (source et cible mélangées), `Union-P` (idem, avec autant de poids pour la cible que pour la source) et `Oracle` (toute la réserve cible étiquetée : le plafond).

**Les méthodes de transfert** (famille C de l'état de l'art : on **retouche les arbres de la forêt source** avec les exemples cibles étiquetés ; code dans `transfer_forest/methods/segev.py`). Un arbre est une suite de questions (« `x2bar` ≤ 4 ? ») qui mène à une feuille, c'est-à-dire à une réponse.

- **SER** (Segev et al., 2017) : *étendre puis élaguer*. Les exemples cibles descendent dans chaque arbre source. Une feuille qui reçoit des lettres de classes différentes est remplacée par un petit arbre appris sur ces lettres (expansion). Puis, de bas en haut, un bloc de questions qui ne fait pas mieux qu'une réponse simple est remplacé par une feuille (réduction). *Exemple : une feuille dit « A » ; trois lettres larges y arrivent, un H, un N et un A ; SER y greffe un mini-arbre qui les sépare.*
- **STRUT** (Segev et al., 2017) : *garder la structure, déplacer les seuils*. Mêmes variables, mêmes questions, mais chaque seuil est recalculé sur les exemples cibles : parmi les meilleurs candidats, on garde celui dont les deux côtés ressemblent le plus (divergence de Jensen-Shannon) à ceux de la source. Les branches qu'aucun exemple cible n'atteint sont élaguées. *Exemple : la question « `x2bar` ≤ 4 ? » devient « `x2bar` ≤ 9 ? ».*
- **SER\*** (Minvielle et al., 2019) : SER, mais un nœud n'est **jamais élagué si la classe rare existait à ce nœud dans la source**. Cela évite d'effacer une branche qui sait reconnaître une lettre rare, parce que les quelques exemples cibles qui y arrivent sont d'une autre classe. La classe rare est `ctx.rare_class`, sinon la moins fréquente de l'échantillon cible étiqueté.
- **STRUT\*** (Minvielle et al., 2019) : STRUT, mais les proportions de classes de la source sont **corrigées par $\lambda_k = P_T(k)/P_S(k)$** avant la comparaison, pour tenir compte du changement de fréquence des lettres (le O est 41 fois plus fréquent dans la cible).

**Points communs.** Aucune de ces méthodes ne relit les données source : elles n'utilisent que la forêt source (et `ys`, seulement pour estimer $P_S(Y)$ dans STRUT\*). Une feuille prédit les proportions de classes des exemples cibles qui y arrivent, ou la distribution de la source si aucun n'y arrive ; la forêt moyenne les probabilités de ses arbres.

**À quoi s'attendre** (état de l'art, section 4) : SER l'emporte quand la correspondance entre variables n'est pas conservée (jeux de pixels, donc S1) ; STRUT, quand seuls les seuils se déplacent. Dans un test sans aucun décalage, STRUT retombait au niveau de `Tgt` : ses feuilles ne reposent que sur les exemples cibles.

In [ ]:
RESULTS_FILE = Path.cwd().parent / "results" / "s1_transfert.csv"   # une ligne par résultat ; dossier ignoré par git
N_JOBS = -1                                                          # tous les cœurs ; 1 = pas de parallélisme

def error(name, ctx):
    P = make_method(name).fit(ctx).predict_proba(X_test)
    return (P.argmax(axis=1) != y_test).mean()

def run_seed(seed, todo):
    """Les couples (méthode, budget) demandés pour une graine : une ligne de résultat chacun."""
    sys.path.insert(0, str(RESULTS_FILE.parents[1]))                 # dans un processus fils : retrouver transfer_forest
    rf_src = make_forest(seed).fit(scn.Xs, scn.ys)                   # forêt source de la graine, partagée par toutes les méthodes
    order = reserve_order(seed)
    rows = []
    for name, p in todo:
        lab, unlab = order[:n_T[p]], order[n_T[p]:]
        ctx = Context(scn.Xs, scn.ys, Xr[lab], yr[lab], Xr[unlab], rf_src, scn.K, seed,
                      X_reserve=Xr, y_reserve=yr)
        rows.append({"seed": seed, "n_T": n_T[p] if METHODS[name].uses_budget else 0,   # 0 : Src et Oracle, sans budget
                     "method": name, "err": error(name, ctx)})
    return rows

# ce qui est déjà calculé, puis ce qu'il reste à calculer (Src et Oracle : une fois par graine)
done = pd.read_csv(RESULTS_FILE) if RESULTS_FILE.exists() else pd.DataFrame(columns=["seed", "n_T", "method", "err"])
done_keys = set(zip(done.seed, done.n_T, done.method))
wanted = {}
for seed in SEEDS:
    for p in BUDGETS:
        for name, cls in METHODS.items():
            if cls.uses_budget or p == BUDGETS[0]:
                if (seed, n_T[p] if cls.uses_budget else 0, name) not in done_keys:
                    wanted.setdefault(seed, []).append((name, p))
print(sum(len(v) for v in wanted.values()), "expériences à calculer sur", len(wanted), "graine(s)")

if wanted:
    RESULTS_FILE.parent.mkdir(exist_ok=True)
    jobs = Parallel(n_jobs=N_JOBS, return_as="generator_unordered")(
        delayed(run_seed)(seed, todo) for seed, todo in wanted.items())
    for rows in jobs:                                                # on enregistre après chaque graine terminée
        new = pd.DataFrame(rows)
        done = new if done.empty else pd.concat([done, new], ignore_index=True)
        done.to_csv(RESULTS_FILE, index=False)
        print("graine", rows[0]["seed"], "terminée")

results = done[done.seed.isin(list(SEEDS))].reset_index(drop=True)
results["n_T"] = results.n_T.replace(0, np.nan)                      # Src et Oracle : n_T vide, comme dans la première expérience
print(len(results), "expériences")

## 5. Les résultats

Taux d'erreur sur le test cible, moyenne et écart type sur les 20 graines (`n_T` vide : Src et Oracle, indépendantes du budget).

In [ ]:
results.groupby(["method", "n_T"], dropna=False)["err"].agg(["mean", "std"]).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name in [m for m in METHODS if m not in ("Src", "Oracle")]:
    curve = results[results.method == name].groupby("n_T")["err"]
    ax.errorbar(curve.mean().index, curve.mean(), yerr=curve.std(), marker="o", capsize=3, label=name)
for name, style in [("Src", "--"), ("Oracle", ":")]:
    ax.axhline(results.loc[results.method == name, "err"].mean(), color="k", ls=style, label=name)
ax.set(xscale="log", xlabel=r"$n_T$ : exemples cibles étiquetés", ylabel="erreur sur le test cible",
       title=f"S1 : références et méthodes de transfert (moyenne ± écart type, {len(SEEDS)} graines)")
ax.legend()
plt.show()

### Lecture

| Référence | Erreur | Lecture |
|---|---|---|
| Src | $57{,}5\,\%$ ($\pm 1{,}0$) | la forêt source rate plus d'une lettre sur deux dans la cible |
| Oracle | $5{,}9\,\%$ ($\pm 0{,}2$) | avec toute la réserve étiquetée, la cible est facile |
| Tgt | de $40{,}6\,\%$ ($n_T = 108$) à $9{,}4\,\%$ ($n_T = 2\,164$) | s'améliore régulièrement avec le budget |
| Union | de $38{,}4\,\%$ à $8{,}1\,\%$ | à peine mieux que Tgt |
| Union-P | de $31{,}4\,\%$ à $7{,}8\,\%$ | la meilleure référence à tous les budgets |

**Le décalage est réel et massif.** La même forêt source fait $4{,}4\,\%$ d'erreur sur la source (erreur hors sac, section 7) et $57{,}5\,\%$ sur la cible : l'écart de domaine est d'environ $53$ points. Ce n'est pas la cible qui est difficile : Oracle y atteint $5{,}9\,\%$, du même ordre que la source. C'est bien le **décalage** qui fait échouer Src.

**Les ordres de grandeur sont plausibles.** Le guide cite pour Letter, d'après Segev : Src $\approx 66\,\%$, Tgt $\approx 19\,\%$. Notre Tgt fait $18{,}9\,\%$ à $n_T = 541$ (5 %) : ce chiffre de Segev correspond peut-être à ce budget, **à vérifier dans l'article**. Notre Src est plus basse de 8 points : l'écart reste à comprendre (découpage du test, réglages des forêts).

## 6. Comparaisons appariées

Le protocole compare les méthodes **graine par graine** : à graine et budget égaux, elles ont reçu exactement les mêmes exemples. On calcule, pour chaque budget, la proportion des 20 graines où une référence fait mieux que Tgt, et le gain moyen en points d'erreur, $\bar G(p) = \frac{1}{R}\sum_{s=1}^{R} \big(\mathrm{Err}_{\mathrm{Tgt}}(s,p) - \mathrm{Err}_m(s,p)\big)$.

In [ ]:
err = results.pivot_table(index=["seed", "n_T"], columns="method", values="err")
pd.DataFrame({
    "Union-P bat Tgt": (err["Union-P"] < err["Tgt"]).groupby("n_T").mean(),
    "Union bat Tgt": (err["Union"] < err["Tgt"]).groupby("n_T").mean(),
    "gain Union-P (points)": 100 * (err["Tgt"] - err["Union-P"]).groupby("n_T").mean(),
    "gain Union (points)": 100 * (err["Tgt"] - err["Union"]).groupby("n_T").mean(),
}).round(2)

### Lecture

**Union-P bat Tgt sur les 20 graines, à tous les budgets.** Le gain moyen décroît avec le budget : $9{,}2$ points à $n_T = 108$, puis $6{,}2$ ; $3{,}7$ ; $2{,}3$ ; $1{,}6$. C'est le profil attendu d'un transfert utile : la source compte surtout quand la cible est pauvre. À $n_T = 108$, Union-P ($31{,}4\,\%$) fait à peu près comme Tgt avec deux fois plus d'exemples ($29{,}1\,\%$ à $n_T = 216$) : environ **la moitié des annotations économisées**.

**Union n'apporte presque rien.** Gain moyen entre $-0{,}3$ et $+2{,}2$ points, et Union ne bat Tgt que sur 45 % des graines à $n_T = 216$. Explication : sans pondération, la source représente plus de 99 % des données à 1 % de budget ; la forêt apprend surtout la source. Rééquilibrer les masses, comme le fait Union-P,
$$\tfrac12\,\hat P_S + \tfrac12\,\hat P_T \quad\text{au lieu de}\quad \tfrac{n_S}{n_S + n_T}\,\hat P_S + \tfrac{n_T}{n_S + n_T}\,\hat P_T ,$$
change tout.

**La variabilité compte.** À petit budget, l'écart type de Tgt atteint 2 points, plus que certains écarts entre méthodes (Union contre Tgt). Seules les comparaisons appariées, puis le test de Wilcoxon prévu au protocole, permettront de conclure sur des écarts de cette taille.

### Les méthodes de transfert, mêmes comparaisons

Même calcul que ci-dessus, pour chaque méthode de transfert de `METHODS` : d'abord contre **Tgt** (la référence principale), puis contre **Union-P** (la référence la plus forte). « bat » : proportion des graines où la méthode fait mieux ; gain moyen en points d'erreur, positif = la méthode fait mieux.

In [ ]:
NEW = [m for m in METHODS if m not in ("Src", "Tgt", "Union", "Union-P", "Oracle")]
pd.DataFrame({col: val for m in NEW for col, val in {
    f"{m} bat Tgt": (err[m] < err["Tgt"]).groupby("n_T").mean(),
    f"gain {m} (points)": 100 * (err["Tgt"] - err[m]).groupby("n_T").mean(),
}.items()}).round(2)

In [ ]:
pd.DataFrame({col: val for m in NEW for col, val in {
    f"{m} bat Union-P": (err[m] < err["Union-P"]).groupby("n_T").mean(),
    f"gain {m} (points)": 100 * (err["Union-P"] - err[m]).groupby("n_T").mean(),
}.items()}).round(2)

### Lecture (méthodes de transfert)

Erreur moyenne sur le test cible (%), 20 graines. Les écarts ci-dessous sont des **gains moyens par graine, en points d'erreur** (positif = la méthode fait mieux) ; « k/20 » = nombre de graines où la méthode bat la référence. Test de Wilcoxon apparié : avec 20 graines, la plus petite p-valeur possible est $\approx 2\cdot10^{-6}$ (toutes les graines dans le même sens).

| $n_T$ | Tgt | Union-P | SER | SER\* | STRUT | STRUT\* |
|---|---|---|---|---|---|---|
| 108 | 40,6 | 30,9 | 33,4 | 32,0 | 54,0 | 50,3 |
| 216 | 29,1 | 22,6 | 24,9 | 24,0 | 47,3 | 44,6 |
| 541 | 18,9 | 14,6 | 16,4 | 16,0 | 39,7 | 36,6 |
| 1 082 | 13,4 | 10,4 | 11,4 | 11,2 | 36,5 | 34,1 |
| 2 164 | 9,4 | 7,3 | 7,8 | 7,7 | 35,6 | 33,2 |

**1. SER et SER\* battent Tgt, partout.** SER fait mieux que Tgt sur 20 graines sur 20 à tous les budgets : $+7{,}3$ ; $+4{,}3$ ; $+2{,}5$ ; $+2{,}0$ ; $+1{,}6$ points (p $\le 10^{-4}$). SER\* fait un peu mieux : $+8{,}7$ à $+1{,}7$ points. Le gain décroît avec le budget : la source compte surtout quand la cible est pauvre.

**2. Aucune méthode de transfert ne bat Union-P.** SER ne bat Union-P que sur 5, 1, 0, 0 et 2 graines sur 20 (gains moyens $-2{,}4$ ; $-2{,}3$ ; $-1{,}7$ ; $-1{,}1$ ; $-0{,}5$). SER\* est un peu plus proche ($-1{,}0$ à $-1{,}4$ ; à $n_T = 108$, 7 graines sur 20 et p $= 0{,}05$ : la différence n'est pas significative), mais ne gagne pas non plus. L'écart fond avec le budget : à $n_T = 2\,164$, Union-P ne devance SER que de $0{,}5$ point. Une nuance à garder : Union-P **relit les données source**, ce que SER n'a pas le droit de faire (il n'a que la forêt source). La barre est donc plus haute pour SER que pour une méthode qui aurait accès à la source.

**3. SER\* fait mieux que SER, mais ce n'est pas encore la preuve que la protection de la classe rare marche.** SER\* bat SER sur 20 graines sur 20 à 108 et 216 exemples ($+1{,}4$ et $+0{,}8$ point), puis l'écart se resserre ($+0{,}1$ point à 2 164). Or S1 n'a pas de classe rare : la classe protégée est la moins fréquente de l'échantillon cible étiqueté, un choix arbitraire. Le gain mesure « élaguer moins », pas « sauver une lettre rare ». À refaire avec `rare_class` fixé explicitement (par exemple O, quasi absente de la source) ou sur un scénario conçu pour.

**4. STRUT et STRUT\* perdent contre Tgt sur 20 graines sur 20, à tous les budgets**, de $13$ à $26$ points pour STRUT. Et l'écart **s'aggrave** avec le budget : Tgt passe de $40{,}6$ % à $9{,}4$ %, STRUT reste autour de $36$ %. C'est cohérent avec ce que fait la méthode : elle garde la structure des arbres source et ne déplace que les seuils, donc elle ne peut pas créer les distinctions nouvelles que demande S1 (une lettre étalée change d'apparence : la relation entre variables et classes n'est plus la même). L'état de l'art le prévoyait : SER l'emporte quand la correspondance entre variables n'est pas conservée. STRUT\* fait 2,4 à 3,7 points de mieux que STRUT (17 à 20 graines sur 20) : corriger les proportions de classes aide, comme annoncé en section 3, mais ne rattrape pas Tgt.

**Est-ce un défaut d'implémentation ?** Deux vérifications, **hors de ce notebook** : (a) sur l'exemple synthétique « moving source » de Segev et al. (tableau 1), notre STRUT fait $6{,}4$ % d'erreur contre $6{,}1$ % dans l'article, et bat SER : il fonctionne quand seuls les seuils bougent ; (b) sur Letter coupé *lettre par lettre* (médiane de `x2bar` calculée dans chaque classe, ce qui supprime le décalage des classes), STRUT tombe à $22$ % à 5 % de budget (3 graines, un seul budget) au lieu de $40$ %. Le mauvais score de STRUT sur S1 tient donc en grande partie au **décalage des classes** de la coupe globale. Cela rend aussi la comparaison avec l'article délicate : à 5 %, Segev donne STRUT $21{,}0$ %, SER $18{,}9$ %, Tgt $19{,}3$ % ; nous trouvons STRUT $39{,}7$ %, SER $16{,}4$ %, Tgt $18{,}9$ %. La coupe exacte de l'article n'est pas établie (« médiane pour chaque lettre » dans le texte, mais ses $10\,822$ exemples source correspondent à la coupe globale).

**Précautions.**
- Un seul découpage test/réserve : les 20 graines ne font varier que le tirage des exemples étiquetés et la forêt, pas le test. Les p-valeurs sous-estiment l'incertitude ; lisez-les comme un ordre de grandeur.
- L'écart-type entre graines atteint $2{,}5$ points à $n_T = 108$ (SER). Seules les comparaisons appariées permettent de conclure sur des écarts de cette taille.
- SER élague avec $\mathrm{LErr}(v) \le \mathrm{STErr}(v)$ (égalité comprise). Avec une inégalité stricte, comme l'écrit le pseudo-code de Segev et al., plus rien n'est jamais élagué après l'expansion (le sous-arbre a alors 0 erreur sur les exemples cibles) et SER\* devient identique à SER.

**À retenir.** Sur S1, seul SER fait mieux que Tgt, mais il reste derrière Union-P. L'idée « garder la forêt source et la retoucher » ne bat pas, ici, la simple repondération des données source ; le choix de la coupe de S1 (globale ou lettre par lettre) change le verdict sur STRUT.

## 7. Où Src se trompe-t-elle ?

Pour la graine 0 : l'erreur de la forêt source sur la source (hors sac) et sur la cible, puis son erreur lettre par lettre, mise en regard du changement de proportion $\lambda_k$.

In [ ]:
rf_src = make_forest(0).set_params(oob_score=True).fit(scn.Xs, scn.ys)
pred = rf_src.predict(X_test)
err_k = np.array([(pred[y_test == k] != k).mean() for k in range(scn.K)])
print(f"Src, erreur sur la source (hors sac) : {1 - rf_src.oob_score_:.3f}   sur la cible : {(pred != y_test).mean():.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(lam, err_k)
for k, name in enumerate(scn.class_names):
    ax.annotate(name, (lam[k], err_k[k]), xytext=(4, 2), textcoords="offset points")
ax.set(xscale="log", xlabel=r"$\lambda_k = P_T(k)\,/\,P_S(k)$", ylabel="erreur de Src sur la lettre (test cible)",
       title=f"Src lettre par lettre (corrélation avec log λ : {np.corrcoef(np.log(lam), err_k)[0, 1]:.2f})")
plt.show()

### Lecture

L'erreur de Src varie énormément d'une lettre à l'autre : de $0\,\%$ (X) et $2\,\%$ (J) à $85\,\%$ (G), $94\,\%$ (H) et $98\,\%$ (O). Elle est nettement liée au changement de proportion (corrélation de $0{,}52$ avec $\log \lambda_k$) : les lettres presque absentes de la source, comme O ou G, sont celles que la forêt source ne sait pas reconnaître une fois étalées.

Mais la proportion n'explique pas tout : H est aussi fréquente dans les deux domaines ($\lambda \approx 1{,}5$) et pourtant ratée à $94\,\%$. De même, A et Y, presque absentes de la cible ($\lambda < 0{,}2$), y sont ratées à $68\,\%$ et $77\,\%$. Pour ces lettres, c'est l'**apparence** qui change quand la lettre s'élargit : la règle $P(Y \mid X)$ elle-même se déplace. S1 mêle donc les trois formes de décalage de l'état de l'art : covariables, classes et, localement, concept.

## 8. Bilan et regard critique

1. **Le décalage de S1 est réel et massif** : environ 53 points d'écart de domaine. Ce n'est pas la tâche cible qui est difficile (Oracle : $5{,}9\,\%$), c'est le passage d'un domaine à l'autre.
2. **Ce décalage est mixte** : covariables (supports disjoints sur `x2bar`), classes ($\lambda_k$ de $0{,}08$ à $41$) et, pour certaines lettres comme H, concept. La description du protocole (« décalage des entrées ») est à compléter, et SER*/STRUT* méritent d'être testés sur S1.
3. **Union-P est une référence forte** : meilleure que Tgt sur 20 graines sur 20, à tous les budgets, avec environ moitié moins d'annotations à petit budget. La barre est haute : **une méthode de transfert devra battre Union-P, pas seulement Tgt**.
4. **Conséquence pour le protocole** : le gain $G_{\mathrm{ref}} = \min(\mathrm{Err}_{\mathrm{Src}}, \mathrm{Err}_{\mathrm{Tgt}}, \mathrm{Err}_{\mathrm{Union}}) - \mathrm{Err}_m$ ne contient pas Union-P, qui est ici la meilleure référence. Une méthode pourrait afficher un gain positif tout en faisant moins bien qu'une simple repondération. **Il faudrait ajouter Union-P à ce minimum.**
5. **La cible est déséquilibrée** : de 47 à plus de 700 exemples par lettre. Le taux d'erreur reste la note principale (comme Segev), mais la BER, prévue en note secondaire, sera utile pour vérifier que les gains ne portent pas que sur les lettres fréquentes.

**Limites de cette première expérience.** Tirage et notation provisoires, codés dans ce notebook ; un seul découpage test/réserve ; hyperparamètres par défaut ; pas encore de test statistique. Les chiffres de Segev restent à vérifier dans l'article (budget correspondant, écart de 8 points sur Src).